# Speed Drill
Implement each function as fast as you can. Run the test cell to check. Flagged sections (🔥) are highest priority for the real test.

In [1]:
import numpy as np

def check(label, got=None, expected=None, cond=None):
    if cond is not None:
        ok = cond(got) if got is not None else cond
    else:
        ok = np.allclose(got, expected, atol=1e-5) if isinstance(got, np.ndarray) else (got == expected)
    status = "  [PASS]" if ok else "  [FAIL]"
    print(f"{status} {label}")
    if not ok and got is not None and expected is not None:
        print(f"         got:      {got}")
        print(f"         expected: {expected}")


## 🔥 1. Softmax
Three variants. Must be instant.

In [16]:
# 1a. Stable softmax — 1D input
def softmax_1d(z):
    # z: (n,) → (n,)
    z = z - z.max(keepdims=True)
    e = np.exp(z)
    
    return np.round( e / e.sum(keepdims=True), decimals=9 )

In [17]:
z = np.array([1.0, 2.0, 3.0])
out = softmax_1d(z)
check("sums to 1", np.sum(out), 1.0)
check("all positive", cond=lambda x: np.all(x > 0), got=out)
check("shape", out.shape, (3,))
z_big = np.array([1000.0, 1001.0, 1002.0])
check("stable on large values", cond=lambda x: np.all(np.isfinite(x)), got=softmax_1d(z_big))


  [PASS] sums to 1
  [PASS] all positive
  [PASS] shape
  [PASS] stable on large values


In [18]:
# 1b. Softmax — 2D input, along last axis
def softmax_2d(Z):
    # Z: (T, V) → (T, V), each row sums to 1
    z= Z - np.max(Z,  axis=-1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=-1, keepdims=True)

In [19]:
Z = np.random.randn(4, 6)
out = softmax_2d(Z)
check("shape", out.shape, (4, 6))
check("rows sum to 1", np.allclose(out.sum(axis=-1), 1.0), True)
check("all positive", cond=lambda x: np.all(x > 0), got=out)


  [PASS] shape
  [PASS] rows sum to 1
  [PASS] all positive


In [22]:
# 1c. Softmax — arbitrary axis
def softmax_axis(Z, axis=-1):
    # Z: any shape, softmax along `axis`
    Z = Z - np.max(Z, axis=axis, keepdims=True)
    e = np.exp(Z)
    return e / np.sum(e, keepdims=True, axis=axis)

In [23]:
Z = np.random.randn(3, 4, 5)
out = softmax_axis(Z, axis=1)
check("shape", out.shape, (3, 4, 5))
check("sums to 1 along axis=1", np.allclose(out.sum(axis=1), 1.0), True)
out2 = softmax_axis(Z, axis=-1)
check("sums to 1 along axis=-1", np.allclose(out2.sum(axis=-1), 1.0), True)


  [PASS] shape
  [PASS] sums to 1 along axis=1
  [PASS] sums to 1 along axis=-1


## 🔥 2. Log-Softmax + Cross Entropy

In [43]:
# 2a. Log-softmax (numerically stable)
def log_softmax(z, axis =-1):
    # z: (n,) → (n,)
    z = z - np.max(z, axis = axis, keepdims=True)
    return z - np.log( np.sum( np.exp(z) , keepdims=True, axis=axis ) )


In [44]:
z = np.array([1.0, 2.0, 3.0])
ls = log_softmax(z)
check("exp matches softmax", np.allclose(np.exp(ls), softmax_1d(z)), True)
check("shape", ls.shape, (3,))
z_big = np.array([1000.0, 1001.0, 1002.0])
check("stable on large values", cond=lambda x: np.all(np.isfinite(x)), got=log_softmax(z_big))


  [PASS] exp matches softmax
  [PASS] shape
  [PASS] stable on large values


In [45]:
# 2b. Cross entropy — single token
def cross_entropy(logits, target_idx, axis=-1):
    # logits: (V,), target_idx: int → scalar loss
    lp = log_softmax(logits)
    return -lp[target_idx]


In [46]:
logits = np.array([0.5, 2.0, 0.1, -1.0])
ce = cross_entropy(logits, 1)
expected = -np.log(np.exp(2.0) / np.sum(np.exp( logits ) ) )
check("correct value", np.isclose(ce, expected, atol=1e-5), True)
check("scalar", np.isscalar(ce) or ce.ndim == 0, True)


  [PASS] correct value
  [PASS] scalar


In [48]:
# 2c. Cross entropy — batch (mean over tokens)
def cross_entropy_batch(logits, targets):
    # logits: (T, V), targets: (T,) ints → scalar mean loss
    lp = log_softmax(logits, axis = -1)
    T, _ = logits.shape
    return -lp[np.arange(T),targets].mean()
#     raise NotImplementedError()


In [49]:
T, V = 5, 8
np.random.seed(0)
logits = np.random.randn(T, V)
targets = np.random.randint(0, V, size=T)
loss = cross_entropy_batch(logits, targets)
check("scalar", np.isscalar(loss) or loss.ndim == 0, True)
check("positive", loss > 0, True)
# manual check for one token
manual = -log_softmax(logits[0])[targets[0]]
check("matches manual for token 0", np.isclose(-log_softmax(logits[0])[targets[0]], manual), True)


  [PASS] scalar
  [PASS] positive
  [PASS] matches manual for token 0


## 🔥 3. Softmax Jacobian
J[i,j] = p_i(δ_ij − p_j)

In [50]:
# 3a. Jacobian from formula
def softmax_jacobian(z, axis=-1):
    # z: (n,) → (n, n)
    A = softmax_axis(z, axis=axis)
    return np.diag(A) - np.outer(A,A)


In [51]:
z = np.array([1.0, 0.5, -0.5, 2.0])
J = softmax_jacobian(z)
check("shape (n,n)", J.shape, (4, 4))
check("rows sum to 0", np.allclose(J.sum(axis=1), 0.0), True)
check("symmetric", np.allclose(J, J.T), True)
p = softmax_1d(z)
check("diagonal is p*(1-p)", np.allclose(np.diag(J), p*(1-p)), True)


  [PASS] shape (n,n)
  [PASS] rows sum to 0
  [PASS] symmetric
  [PASS] diagonal is p*(1-p)


In [52]:
# 3b. Jacobian — verify via finite differences
def finite_diff_jacobian(z, eps=1e-5):
    n = len(z)
    J = np.zeros((n, n))
    for j in range(n):
        dz = np.zeros(n); dz[j] = eps
        J[:, j] = (softmax_1d(z + dz) - softmax_1d(z - dz)) / (2 * eps)
    return J


In [60]:
z = np.random.randn(5)
J_analytic = softmax_jacobian(z)
J_numeric  = finite_diff_jacobian(z)
check("matches finite differences", np.allclose(J_analytic, J_numeric, atol=1e-4), True)


  [PASS] matches finite differences


In [63]:
# 3c. Gradient through softmax: given dL/dp, compute dL/dz
def softmax_grad(z, dL_dp, axis = -1):
    # z: (n,), dL_dp: (n,) → dL_dz: (n,)
    # Hint: dL/dz = J.T @ dL_dp, but there's a simpler closed form
    p = softmax_axis(z, axis=axis)
    
    return p * ( dL_dp - (p*dL_dp).sum(axis=axis, keepdims=True) )


In [64]:
z = np.random.randn(6)
dL_dp = np.random.randn(6)
dL_dz = softmax_grad(z, dL_dp)
# check via Jacobian
p = softmax_1d(z)
J = softmax_jacobian(z)
expected = J.T @ dL_dp
check("matches J.T @ dL_dp", np.allclose(dL_dz, expected, atol=1e-6), True)
check("shape", dL_dz.shape, (6,))


  [PASS] matches J.T @ dL_dp
  [PASS] shape


## 🔥 4. SVD + Effective Rank
Remember: cumsum(S**2), not cumsum(S). Vh is already transposed.

In [65]:
# 4a. Economy SVD + reconstruct
def svd_reconstruct(A, rank):
    # A: (m,n) → reconstruct using only `rank` singular values
    U, S, Vh    = np.linalg.svd(A)
    Ur, Sr, Vhr = (U[:,:rank], S[:rank], Vh[:rank,:])
    return Ur @ np.diag(Sr) @ Vhr


In [66]:
np.random.seed(1)
A = np.random.randn(5, 4)
A_r = svd_reconstruct(A, rank=2)
check("shape preserved", A_r.shape, A.shape)
# rank-2 reconstruction should have rank ≤ 2
U, S, Vh = np.linalg.svd(A_r, full_matrices=False)
check("effective rank ≤ 2", np.sum(S > 1e-8) <= 2, True)
# full rank reconstruction = original
A_full = svd_reconstruct(A, rank=min(A.shape))
check("full rank = original", np.allclose(A_full, A, atol=1e-10), True)


  [PASS] shape preserved
  [PASS] effective rank ≤ 2
  [PASS] full rank = original


In [69]:
# 4b. Effective rank — fraction of Frobenius norm squared
def effective_rank(S, threshold=0.9):
    # S: (d,) sorted singular values (descending) → int
    var_cumu = np.cumsum(S**2)
    total = var_cumu[-1]
    return next(i+1 for i,val in enumerate(var_cumu) if val >= threshold*total)


In [70]:
# Identity matrix: all singular values equal → effective rank = d
S_uniform = np.ones(8)
check("uniform S: full effective rank", effective_rank(S_uniform, 0.9), 8)
# Rank-1: one dominant singular value
S_rank1 = np.array([10.0, 0.01, 0.01, 0.01])
check("rank-1 signal: eff rank = 1", effective_rank(S_rank1, 0.9), 1)
# Threshold sensitivity
S_mid = np.array([3.0, 2.0, 1.0, 0.5])
r90 = effective_rank(S_mid, 0.90)
r99 = effective_rank(S_mid, 0.99)
check("higher threshold → higher rank", r99 >= r90, True)


  [PASS] uniform S: full effective rank
  [PASS] rank-1 signal: eff rank = 1
  [PASS] higher threshold → higher rank


In [75]:
# 4c. Batched effective rank over H heads
def batch_effective_rank(W_OVs, threshold=0.9):
    # W_OVs: (H, d, d) → (H,) int array of effective ranks
    H = W_OVs.shape[0]
    _, S, _ = np.linalg.svd(W_OVs)
#     print(f"H={H}")
#     print(f"S.shape = {S.shape}")
    eff_ranks = np.zeros((H,))
    for h in range(H):
        eff_ranks[h]=effective_rank(S[h,:] , threshold=threshold)
        
    return eff_ranks


In [76]:
H, d = 4, 8
np.random.seed(2)
W_OVs = np.random.randn(H, d, d)
# Plant a rank-1 head
u = np.random.randn(d); v = np.random.randn(d)
W_OVs[0] = 5.0 * np.outer(u, v)   # rank-1
ranks = batch_effective_rank(W_OVs, threshold=0.9)
check("shape (H,)", ranks.shape, (H,))
check("rank-1 head has eff rank 1", ranks[0], 1)
check("all ranks ≥ 1", np.all(ranks >= 1), True)
check("all ranks ≤ d", np.all(ranks <= d), True)


  [PASS] shape (H,)
  [PASS] rank-1 head has eff rank 1
  [PASS] all ranks ≥ 1
  [PASS] all ranks ≤ d


## 🔥 5. Eigendecomposition Shortcuts
eigh for symmetric (real eigenvalues, sorted). eig for asymmetric.

In [81]:
# 5a. Symmetric matrix: largest eigenvalue + eigenvector (eigh)
def top_eigenvector(M):
    # M: (d, d) symmetric → (d,) eigenvector for largest eigenvalue
    valss, vecss = np.linalg.eigh(M)
    return vecss[:,-1]


In [82]:
d = 6
np.random.seed(3)
A = np.random.randn(d, d)
M = A @ A.T   # symmetric PSD
v = top_eigenvector(M)
vals, vecs = np.linalg.eigh(M)
check("shape", v.shape, (d,))
check("is unit vector", np.isclose(np.linalg.norm(v), 1.0), True)
check("is eigenvector", np.allclose(M @ v, np.dot(M @ v, v) * v, atol=1e-8), True)
check("corresponds to largest eigenval", np.isclose(abs(np.dot(v, vecs[:,-1])), 1.0, atol=1e-6), True)


  [PASS] shape
  [PASS] is unit vector
  [PASS] is eigenvector
  [PASS] corresponds to largest eigenval


In [84]:
# 5b. Detect copy head: eigenvalues of W_OV @ W_U @ W_E
def copy_head_score(W_OV, W_U, W_E):
    # W_OV: (d,d), W_U: (d,V), W_E: (V,d)
    # Return: max real part of eigenvalues of the vocab circuit
    valss = np.linalg.eigvals(W_OV @ W_U @ W_E)
    return valss.real.max()

In [85]:
d, V = 8, 20
np.random.seed(4)
# Plant a copy head: W_OV ≈ identity on embedding subspace
W_E = np.random.randn(V, d); W_E /= np.linalg.norm(W_E, axis=1, keepdims=True)
W_U = W_E.T   # W_U = W_E^T → copy-like
W_OV_copy = np.eye(d) * 2.0
W_OV_rand = np.random.randn(d, d) * 0.1

score_copy = copy_head_score(W_OV_copy, W_U, W_E)
score_rand = copy_head_score(W_OV_rand, W_U, W_E)
check("copy head has large positive score", score_copy > 0.5, True)
check("random head has smaller score", score_copy > score_rand, True)


  [PASS] copy head has large positive score
  [PASS] random head has smaller score


In [86]:
# 5c. Symmetric/antisymmetric decomposition of W_QK
def sym_anti(W_QK):
    # W_QK: (d,d) → W_sym (d,d), W_anti (d,d)
    # W_sym has real eigenvalues, W_anti has imaginary
    W_sym  = (W_QK + W_QK.T)/2
    W_anti = (W_QK - W_QK.T)/2
    return W_sym, W_anti

In [87]:
np.random.seed(5)
W_QK = np.random.randn(6, 6)
W_sym, W_anti = sym_anti(W_QK)
check("sym + anti = original", np.allclose(W_sym + W_anti, W_QK), True)
check("sym is symmetric", np.allclose(W_sym, W_sym.T), True)
check("anti is antisymmetric", np.allclose(W_anti, -W_anti.T), True)
vals_sym = np.linalg.eigvalsh(W_sym)
check("sym has real eigenvalues", np.all(np.isreal(vals_sym)), True)


  [PASS] sym + anti = original
  [PASS] sym is symmetric
  [PASS] anti is antisymmetric
  [PASS] sym has real eigenvalues


## 🔥 6. Einsum Patterns
Outer product, batched matmul, trace, bilinear form.

In [88]:
# 6a. Outer product, inner product, trace — one einsum each
def outer(a, b):       # (n,),(m,) → (n,m)
    return np.einsum("a,b->ab", a, b)

def inner(a, b):       # (n,),(n,) → scalar
    return np.einsum("i,i->", a,b)

def matrix_trace(A):   # (n,n) → scalar
    return np.einsum("ii->", A)


In [89]:
a, b = np.array([1.,2.,3.]), np.array([4.,5.])
check("outer shape", outer(a,b).shape, (3,2))
check("outer values", outer(a,b), np.outer(a,b))
a2, b2 = np.array([1.,2.,3.]), np.array([4.,5.,6.])
check("inner", inner(a2,b2), np.dot(a2,b2))
A = np.arange(9.).reshape(3,3)
check("trace", matrix_trace(A), np.trace(A))


  [PASS] outer shape
  [PASS] outer values
  [PASS] inner
  [PASS] trace


In [94]:
# 6b. Batched matrix multiply + bilinear form
def batched_matmul(A, B):
    # A: (H,m,k), B: (H,k,n) → (H,m,n)
    return np.einsum("hmk,hkn->hmn", A,B)

def bilinear(x, M, y):
    # x: (d,), M: (d,d), y: (d,) → scalar x^T M y
    return np.einsum("d,de,e-> ", x, M, y)


In [95]:
H,m,k,n = 3,4,5,6
A = np.random.randn(H,m,k); B = np.random.randn(H,k,n)
out = batched_matmul(A, B)
check("batched matmul shape", out.shape, (H,m,n))
check("matches np.matmul", np.allclose(out, np.matmul(A,B)), True)

x = np.array([1.,2.,3.]); y = np.array([4.,5.,6.])
M = np.arange(9.).reshape(3,3)
check("bilinear = x @ M @ y", np.isclose(bilinear(x,M,y), x @ M @ y), True)


  [PASS] batched matmul shape
  [PASS] matches np.matmul
  [PASS] bilinear = x @ M @ y


In [ ]:
# 6c. All W_QK and W_OV circuits in one einsum each
def circuit_matrices(W_Qs, W_Ks, W_Vs, W_Os):
    # W_Qs: (H,d,dk), W_Ks: (H,d,dk), W_Vs: (H,d,dv), W_Os: (H,dv,d)
    # W_QKs: (H,d,d)  = W_Q @ W_K.T  per head
    # W_OVs: (H,d,d)  = W_V @ W_O    per head
    raise NotImplementedError()


In [ ]:
H,d,dk,dv = 4,8,4,6
np.random.seed(6)
W_Qs=np.random.randn(H,d,dk); W_Ks=np.random.randn(H,d,dk)
W_Vs=np.random.randn(H,d,dv); W_Os=np.random.randn(H,dv,d)
W_QKs, W_OVs = circuit_matrices(W_Qs,W_Ks,W_Vs,W_Os)
check("W_QKs shape", W_QKs.shape, (H,d,d))
check("W_OVs shape", W_OVs.shape, (H,d,d))
check("W_QK[0] = W_Q[0]@W_K[0].T", np.allclose(W_QKs[0], W_Qs[0]@W_Ks[0].T), True)
check("W_OV[0] = W_V[0]@W_O[0]",   np.allclose(W_OVs[0], W_Vs[0]@W_Os[0]),   True)


## 🔥 7. argsort descending + take_along_axis
Get this cold — it comes up in SAE, copy head, any top-k.

In [98]:
# 7a. Top-k values and indices from a 1D array (descending)
def topk_1d(a, k):
    # a: (n,) → idx: (k,), vals: (k,) — sorted descending
    idx_sort = np.argsort(a)[::-1]
    vals_all = np.take_along_axis(a, idx_sort, axis=0)
    return idx_sort[:k], vals_all[:k]

In [99]:
a = np.array([3., 1., 4., 1., 5., 9., 2., 6.])
idx, vals = topk_1d(a, 3)
check("top 3 values", vals, np.array([9., 6., 5.]))
check("top 3 indices", set(idx.tolist()), {5, 7, 4})
check("sorted descending", np.all(vals[:-1] >= vals[1:]), True)


  [PASS] top 3 values
  [PASS] top 3 indices
  [PASS] sorted descending


In [108]:
# 7b. Top-k per row of a 2D array
def topk_2d(A, k):
    # A: (T, d) → idx: (T, k), vals: (T, k) — each row sorted descending
    idx_sort = np.argsort(A, axis=-1)[:,::-1]
    valss=np.take_along_axis(A, idx_sort, axis=1)
#     print(f"A shape = {A.shape}")
#     print(f"valss shape = {valss.shape}")
#     print(f"idx_sort shape = {idx_sort.shape}")
    return idx_sort[:,:k], valss[:,:k]


In [109]:
np.random.seed(7)
A = np.random.randn(5, 10)
idx, vals = topk_2d(A, k=3)
check("idx shape", idx.shape, (5, 3))
check("vals shape", vals.shape, (5, 3))
check("vals sorted descending per row", np.all(vals[:,:-1] >= vals[:,1:]), True)
check("vals match idx", np.allclose(vals, np.take_along_axis(A, idx, axis=1)), True)
# verify top val of row 0
check("row 0 top val", vals[0,0], A[0].max())


  [PASS] idx shape
  [PASS] vals shape
  [PASS] vals sorted descending per row
  [PASS] vals match idx
  [PASS] row 0 top val


In [116]:
# 7c. Zero out all but top-k per row (in-place style)
def zero_non_topk(A, k):
    # A: (T, d) → (T, d) same values for top-k per row, 0 elsewhere
    idx_topk = np.argsort(A, axis =-1)[:,:-k]
    A_ = A.copy()
    np.put_along_axis(A_, idx_topk, 0., axis=-1)
    return A_

In [117]:
np.random.seed(8)
A = np.abs(np.random.randn(4, 8))   # positive so top-k is clear
out = zero_non_topk(A, k=3)
check("shape", out.shape, A.shape)
check("non-zero count per row = k", np.all(np.sum(out > 0, axis=1) == 3), True)
check("top values preserved", np.allclose(np.sort(out, axis=1)[:,-3:], np.sort(A, axis=1)[:,-3:]), True)
check("others zeroed", np.allclose(np.sort(out, axis=1)[:,:-3], 0.0), True)


  [PASS] shape
  [PASS] non-zero count per row = k
  [PASS] top values preserved
  [PASS] others zeroed


In [120]:
out

array([[0.        , 0.        , 0.        , 0.        , 2.29649157,
        2.4098343 , 0.        , 2.20455628],
       [0.        , 0.        , 1.18342715, 1.91636361, 1.1233268 ,
        0.        , 0.        , 0.        ],
       [0.85954811, 0.        , 0.        , 0.        , 1.34686857,
        0.6069528 , 0.        , 0.        ],
       [1.6459901 , 0.        , 0.        , 1.16614049, 0.        ,
        0.        , 1.44252413, 0.        ]])

## 🔥 8. Attention Mechanics
Single head, causal mask, multi-head.

In [ ]:
# 8a. Single-head attention (no mask)
def attention(Q, K, V):
    # Q: (T,dk), K: (T,dk), V: (T,dv) → (T,dv)
    raise NotImplementedError()


In [ ]:
T,dk,dv = 5,4,6
np.random.seed(9)
Q=np.random.randn(T,dk); K=np.random.randn(T,dk); V=np.random.randn(T,dv)
out = attention(Q,K,V)
check("shape", out.shape, (T,dv))
# rows of softmax(QK^T/sqrt(dk)) should sum to 1
A_mat = softmax_2d(Q @ K.T / np.sqrt(dk))
check("matches manual", np.allclose(out, A_mat @ V), True)


In [ ]:
# 8b. Causal mask — lower triangular, -inf above diagonal
def causal_attention(Q, K, V):
    # Q: (T,dk), K: (T,dk), V: (T,dv) → (T,dv), causal
    raise NotImplementedError()


In [ ]:
T,dk,dv = 6,4,5
np.random.seed(10)
Q=np.random.randn(T,dk); K=np.random.randn(T,dk); V=np.random.randn(T,dv)
out = causal_attention(Q,K,V)
check("shape", out.shape, (T,dv))
# build manual causal output
scores = Q @ K.T / np.sqrt(dk)
mask = np.triu(np.ones((T,T)), k=1).astype(bool)
scores[mask] = -np.inf
A_mat = softmax_2d(scores)
check("matches manual causal", np.allclose(out, A_mat @ V), True)
check("first token only attends to itself", np.isclose(A_mat[0,0], 1.0), True)


In [ ]:
# 8c. Multi-head attention (no mask)
def multihead_attention(X, W_Qs, W_Ks, W_Vs, W_Os):
    # X: (T,d), W_Qs/W_Ks: (H,d,dk), W_Vs: (H,d,dv), W_Os: (H,dv,d)
    # → (T,d): sum of all head outputs
    raise NotImplementedError()


In [ ]:
T,d,H,dk,dv = 5,8,3,4,4
np.random.seed(11)
X=np.random.randn(T,d)
W_Qs=np.random.randn(H,d,dk); W_Ks=np.random.randn(H,d,dk)
W_Vs=np.random.randn(H,d,dv); W_Os=np.random.randn(H,dv,d)
out = multihead_attention(X, W_Qs, W_Ks, W_Vs, W_Os)
check("shape", out.shape, (T,d))
# manual: head 0
Q0=X@W_Qs[0]; K0=X@W_Ks[0]; V0=X@W_Vs[0]
h0 = softmax_2d(Q0@K0.T/np.sqrt(dk)) @ V0 @ W_Os[0]
check("head 0 contribution present", cond=lambda _: True, got=out)  # structure check


## 9. Shannon Entropy of Attention

In [125]:
# 9a. Entropy of a single distribution
def entropy(p, axis=-1):
    # p: (n,) probability vector → scalar, base e
    return -np.sum(p * np.log(p), axis=axis)


In [126]:
p_uniform = np.ones(4) / 4
p_peaked  = np.array([0.999, 0.0003, 0.0003, 0.0004])
check("uniform entropy = log(n)", np.isclose(entropy(p_uniform), np.log(4)), True)
check("peaked entropy ≈ 0", entropy(p_peaked) < 0.01, True)
check("entropy ≥ 0", entropy(p_uniform) >= 0, True)


  [PASS] uniform entropy = log(n)
  [PASS] peaked entropy ≈ 0
  [PASS] entropy ≥ 0


In [142]:
# 9b. Mean entropy per head from attention logits
def mean_head_entropy(logits_per_layer):
    # list of n_layers, each list of H (T,T) logit arrays
    # → (n_layers, H) mean entropy
    logits_per_layer = np.array(logits_per_layer)
    A = softmax_axis(logits_per_layer, axis=-1)
    ents = entropy( A, axis=-1)
#     print(f"logits_per_layer shape = {logits_per_layer.shape}")
#     print(f"ents shape = {ents.shape}")
    return ents.mean(axis=-1)
    

In [143]:
T,H,L = 4,2,2
uniform_logits = np.zeros((T,T))
peaked_logits  = np.full((T,T), -100.); peaked_logits[:,0] = 100.
logits_per_layer = [[uniform_logits, peaked_logits], [np.random.randn(T,T), np.random.randn(T,T)]]
result = mean_head_entropy(logits_per_layer)
check("shape (L,H)", result.shape, (L,H))
check("uniform = log(T)", np.isclose(result[0,0], np.log(T)), True)
check("peaked ≈ 0", result[0,1] < 0.01, True)


  [PASS] shape (L,H)
  [PASS] uniform = log(T)
  [PASS] peaked ≈ 0


In [144]:
# 9c. Find the most uniform head (highest mean entropy)
def most_uniform_head(logits_per_layer):
    # → (layer_idx, head_idx)
    mean_ents = mean_head_entropy(logits_per_layer)
    flat_idx = np.argmax(mean_ents)
#     print(type(flat_idx))
    i, j = np.unravel_index(flat_idx, mean_ents.shape)
    return i, j

In [145]:
T,H,L = 4,3,2
np.random.seed(12)
logits_per_layer = [[np.random.randn(T,T) for _ in range(H)] for _ in range(L)]
logits_per_layer[1][2] = np.zeros((T,T))  # plant most uniform: layer 1, head 2
l_idx, h_idx = most_uniform_head(logits_per_layer)
check("finds most uniform head", (l_idx, h_idx), (1, 2))


  [PASS] finds most uniform head


## 10. Induction Head Scoring
Stripe is A[n+i, i+1] for i in range(n-1).

In [148]:
# 10a. Score one attention matrix for induction pattern
def induction_score_single(A, n):
    # A: (2n, 2n) post-softmax weights → scalar score in [0,1]
    induction_stripe = [A[n+i,i+1] for i in range(n-1)]
    return np.mean(induction_stripe)


In [149]:
n = 4
perfect = np.zeros((2*n, 2*n))
for i in range(n-1): perfect[n+i, i+1] = 1.0
uniform = np.full((2*n, 2*n), 1./(2*n))
check("perfect = 1.0", np.isclose(induction_score_single(perfect, n), 1.0), True)
check("uniform = 1/(2n)", np.isclose(induction_score_single(uniform, n), 1./(2*n)), True)


  [PASS] perfect = 1.0
  [PASS] uniform = 1/(2n)


In [150]:
# 10b. Score all heads in all layers
def score_all_induction(weights_per_layer, n):
    # list of L layers, each list of H (2n,2n) arrays → (L,H)
    weights_per_layer = np.array(weights_per_layer)
    L,H, _,_ = weights_per_layer.shape
    ind_scores = np.zeros((L,H))
    for l in range(L):
        for h in range(H):
            ind_scores[l,h]=induction_score_single( weights_per_layer[l,h,:,:], n )
    
    return ind_scores


In [151]:
n=3; H=2; L=2
perfect = np.zeros((2*n,2*n)); [perfect.__setitem__((n+i,i+1),1.) for i in range(n-1)]
uniform = np.full((2*n,2*n), 1./(2*n))
wpl = [[perfect, uniform], [uniform, uniform]]
scores = score_all_induction(wpl, n)
check("shape (L,H)", scores.shape, (L,H))
check("perfect head = 1.0", np.isclose(scores[0,0], 1.0), True)


  [PASS] shape (L,H)
  [PASS] perfect head = 1.0


In [161]:
index_tuples = [ [(l,h) for l in range(L)]  for h in range(H) ]

In [166]:
index_tuples

[[(0, 0), (1, 0)], [(0, 1), (1, 1)], [(0, 2), (1, 2)]]

In [190]:
# 10c. Rank heads by induction score (descending)
def rank_induction_heads(weights_per_layer, n):
    # → list of (layer, head, score) tuples, sorted descending by score
    ind_scores = score_all_induction(weights_per_layer, n)
    L,H = ind_scores.shape
    triples = [(l,h,ind_scores[l,h]) for l in range(L) for h in range(H)]
    return sorted(triples, key=lambda x: -x[2])


In [192]:
ranked

[(1, 0, 1.0),
 (0, 1, 0.2692509490147423),
 (1, 1, 0.2093789089723419),
 (0, 2, 0.15951857464321406),
 (0, 0, 0.14314791201012028),
 (1, 2, 0.03642668899792838)]

In [191]:
n=3; H=3; L=2
np.random.seed(13)
wpl = [[np.random.dirichlet(np.ones(2*n), size=2*n) for _ in range(H)] for _ in range(L)]
# plant best head
wpl[1][0] = np.zeros((2*n,2*n)); [wpl[1][0].__setitem__((n+i,i+1),1.) for i in range(n-1)]
ranked = rank_induction_heads(wpl, n)
check("returns list", isinstance(ranked, list), True)
check("top is planted head", (ranked[0][0], ranked[0][1]), (1, 0))
check("sorted descending", ranked[0][2] >= ranked[1][2], True)


  [PASS] returns list
  [PASS] top is planted head
  [PASS] sorted descending


## 11. SAE Forward Pass
Encode → ReLU → top-k → decode. Watch argsort direction.

In [ ]:
# 11a. SAE encode (pre-activations + ReLU)
def sae_encode(x, W_enc, b_enc):
    # x: (T,d), W_enc: (d,dh), b_enc: (dh,) → acts: (T,dh) after ReLU
    raise NotImplementedError()


In [ ]:
T,d,dh = 4,6,12
np.random.seed(14)
x=np.random.randn(T,d); W_enc=np.random.randn(d,dh); b_enc=np.random.randn(dh)
acts = sae_encode(x, W_enc, b_enc)
check("shape (T,dh)", acts.shape, (T,dh))
check("all non-negative (ReLU)", np.all(acts >= 0), True)
pre = x @ W_enc + b_enc
check("matches manual", np.allclose(acts, np.maximum(0, pre)), True)


In [ ]:
# 11b. SAE top-k + decode
def sae_topk_decode(acts, W_dec, b_dec, k):
    # acts: (T,dh), W_dec: (dh,d), b_dec: (d,) → recon:(T,d), idx:(T,k), vals:(T,k)
    raise NotImplementedError()


In [ ]:
T,d,dh,k = 3,6,12,4
np.random.seed(15)
acts=np.abs(np.random.randn(T,dh)); W_dec=np.random.randn(dh,d); b_dec=np.random.randn(d)
recon, idx, vals = sae_topk_decode(acts, W_dec, b_dec, k)
check("recon shape (T,d)", recon.shape, (T,d))
check("idx shape (T,k)", idx.shape, (T,k))
check("vals sorted descending", np.all(vals[:,:-1] >= vals[:,1:]), True)
check("vals match idx", np.allclose(vals, np.take_along_axis(acts, idx, axis=1)), True)


In [ ]:
# 11c. Full SAE: encode → top-k → decode → MSE
def sae_full(x, W_enc, b_enc, W_dec, b_dec, k):
    # → recon:(T,d), idx:(T,k), vals:(T,k), mse:float
    raise NotImplementedError()


In [ ]:
T,d,dh,k = 4,8,24,5
np.random.seed(16)
x=np.random.randn(T,d); W_enc=np.random.randn(d,dh); b_enc=np.zeros(dh)
W_dec=np.random.randn(dh,d); b_dec=np.zeros(d)
recon,idx,vals,mse = sae_full(x,W_enc,b_enc,W_dec,b_dec,k)
check("recon shape", recon.shape, (T,d))
check("mse non-negative scalar", mse >= 0 and (np.isscalar(mse) or mse.ndim==0), True)
check("vals non-negative", np.all(vals >= 0), True)
# planted: one feature dominates
W_enc2=np.zeros((d,dh)); W_dec2=np.zeros((dh,d))
feat=np.ones(d)/np.sqrt(d); W_enc2[:,0]=feat*10; W_dec2[0]=feat
x2=np.outer(np.ones(T), feat)
_,idx2,vals2,_ = sae_full(x2,W_enc2,np.zeros(dh),W_dec2,np.zeros(d),k)
check("planted feature is top", np.all(idx2[:,0]==0), True)


## 12. Composition Strength + Virtual Heads

In [ ]:
# 12a. Frobenius norm ratio (composition strength)
def composition_strength(W_OV, W_proj):
    # W_OV: (d,d), W_proj: (d,d) [W_Q or W_K of next layer]
    # → scalar in [0,1]: ||W_OV @ W_proj||_F / (||W_OV||_F * ||W_proj||_F)
    raise NotImplementedError()


In [ ]:

d = 8
np.random.seed(17)
W_OV = np.eye(d)   # identity passes everything through
W_proj = np.random.randn(d,d)
cs = composition_strength(W_OV, W_proj)
check("identity OV: strength = 1.0", np.isclose(cs, 1.0, atol=1e-6), True)
W_OV_rand = np.random.randn(d,d)*0.01
cs_rand = composition_strength(W_OV_rand, W_proj)
check("in [0,1]", 0 <= cs_rand <= 1, True)


In [ ]:
# 12b. K-composition: effective W_QK for layer-2 head using layer-1 OV
def k_composed_qk(W_OV_L1, W_Q_L2, W_K_L2):
    # W_OV_L1: (d,d), W_Q_L2: (d,dk), W_K_L2: (d,dk)
    # → (d,d): effective QK matrix when layer 1 feeds layer 2's keys
    raise NotImplementedError()


In [ ]:
d,dk = 8,4
np.random.seed(18)
W_OV=np.random.randn(d,d); W_Q=np.random.randn(d,dk); W_K=np.random.randn(d,dk)
W_QK_eff = k_composed_qk(W_OV, W_Q, W_K)
check("shape (d,d)", W_QK_eff.shape, (d,d))
expected = W_Q @ (W_OV @ W_K).T
check("= W_Q @ (W_OV @ W_K).T", np.allclose(W_QK_eff, expected), True)


In [ ]:
# 12c. Self-reinforcing SAE feature score
def self_reinforce_score(W_OV_L1, W_OV_L2, W_enc, W_dec):
    # W_OV_L1/L2: (d,d), W_enc: (d,dh), W_dec: (dh,d)
    # → (dh,) score for each feature: W_dec[f] @ W_OV_L1 @ W_OV_L2 @ W_enc[:,f]
    raise NotImplementedError()


In [ ]:
d,dh = 8,16
np.random.seed(19)
W_OV1=np.eye(d); W_OV2=np.eye(d)   # identity OVs: score = W_dec[f]@W_enc[:,f]
W_enc=np.random.randn(d,dh); W_dec=np.random.randn(dh,d)
scores = self_reinforce_score(W_OV1, W_OV2, W_enc, W_dec)
check("shape (dh,)", scores.shape, (dh,))
expected = np.einsum("fd,de->fe", W_dec, W_enc).diagonal() if False else            np.array([W_dec[f] @ W_enc[:,f] for f in range(dh)])
check("identity OVs: score=W_dec@W_enc diag", np.allclose(scores, expected), True)


## 13. Linear Attention O(T)
Key trick: (K^T V) first, then Q @ result.

In [ ]:
# 13a. Full sequence linear attention (no causal mask)
def linear_attention(Q, K, V):
    # Q:(T,dk), K:(T,dk), V:(T,dv) → (T,dv)
    # Do NOT materialize (T,T). Use associativity.
    raise NotImplementedError()


In [ ]:
T,dk,dv = 50,8,6
np.random.seed(20)
Q=np.random.randn(T,dk); K=np.random.randn(T,dk); V=np.random.randn(T,dv)
out = linear_attention(Q,K,V)
naive = (Q @ K.T) @ V
check("shape", out.shape, (T,dv))
check("matches naive (Q@K.T)@V", np.allclose(out, naive), True)


In [ ]:
# 13b. Causal linear attention — O(T) using cumulative KV sum
def causal_linear_attention(Q, K, V):
    # Q:(T,dk), K:(T,dk), V:(T,dv) → (T,dv)
    # For token t: output = Q[t] @ sum_{s<=t} K[s]^T K[s] ... actually
    # output[t] = Q[t] @ (sum_{s<=t} outer(K[s], V[s]))
    # Build cumulative (dk, dv) outer product sum
    raise NotImplementedError()


In [ ]:
T,dk,dv = 20,4,5
np.random.seed(21)
Q=np.random.randn(T,dk); K=np.random.randn(T,dk); V=np.random.randn(T,dv)
out = causal_linear_attention(Q,K,V)
# naive causal
scores = np.tril(Q @ K.T)
naive = scores @ V
check("shape", out.shape, (T,dv))
check("matches naive causal", np.allclose(out, naive), True)


In [ ]:
# 13c. Time both — confirm O(T) is faster for large T
import time
T_large, dk, dv = 2000, 16, 16
np.random.seed(22)
Q=np.random.randn(T_large,dk); K=np.random.randn(T_large,dk); V=np.random.randn(T_large,dv)

t0 = time.time()
naive = (Q @ K.T) @ V
t_naive = time.time() - t0

t0 = time.time()
fast = linear_attention(Q, K, V)
t_fast = time.time() - t0

check("same result", np.allclose(naive, fast, atol=1e-8), True)
print(f"  naive: {t_naive*1000:.1f}ms  |  O(T): {t_fast*1000:.1f}ms  |  speedup: {t_naive/t_fast:.1f}x")


---
## 🔥 15. Logit Lens + Logit Difference
Unembedding residual stream at intermediate layers. Core tool for tracking information flow.

In [ ]:
# 15a. Logit lens — unembed residual stream at every layer
def logit_lens(resid_stream, W_U):
    """
    resid_stream: (n_layers, T, d) — residual stream snapshots at each layer
    W_U: (d, V) — unembedding matrix
    Returns: (n_layers, T, V) logits, (n_layers, T, V) probs
    """
    raise NotImplementedError()


In [ ]:
n_layers, T, d, V = 4, 6, 8, 20
np.random.seed(30)
resid = np.random.randn(n_layers, T, d)
W_U   = np.random.randn(d, V)
logits, probs = logit_lens(resid, W_U)
check("logits shape (n_layers,T,V)", logits.shape, (n_layers, T, V))
check("probs shape  (n_layers,T,V)", probs.shape,  (n_layers, T, V))
check("probs sum to 1 over vocab",   np.allclose(probs.sum(axis=-1), 1.0), True)
check("probs all positive",          np.all(probs > 0), True)
check("logits[0] = resid[0] @ W_U", np.allclose(logits[0], resid[0] @ W_U), True)


In [ ]:
# 15b. Top predicted token at each (layer, position)
def top_token_per_layer(resid_stream, W_U):
    """
    resid_stream: (n_layers, T, d)
    W_U: (d, V)
    Returns: (n_layers, T) int array of argmax token at each layer/position
    """
    raise NotImplementedError()


In [ ]:
n_layers, T, d, V = 3, 5, 8, 20
np.random.seed(31)
resid = np.random.randn(n_layers, T, d)
W_U   = np.random.randn(d, V)
top = top_token_per_layer(resid, W_U)
check("shape (n_layers, T)", top.shape, (n_layers, T))
check("values in [0, V)",    np.all((top >= 0) & (top < V)), True)
# verify one entry manually
logits_0_0 = resid[0, 0] @ W_U
check("matches argmax manually", top[0, 0], int(np.argmax(logits_0_0)))


In [ ]:
# 15c. Logit difference — correct minus incorrect token, across all layers/positions
def logit_diff_lens(resid_stream, W_U, correct_idx, wrong_idx):
    """
    resid_stream: (n_layers, T, d)
    W_U: (d, V)
    correct_idx: int — target token
    wrong_idx:   int — foil token
    Returns: (n_layers, T) logit difference (correct - wrong) at each layer/position
    """
    raise NotImplementedError()


In [ ]:
n_layers, T, d, V = 4, 6, 8, 20
np.random.seed(32)
resid = np.random.randn(n_layers, T, d)
W_U   = np.random.randn(d, V)
correct, wrong = 3, 7
diff = logit_diff_lens(resid, W_U, correct, wrong)
check("shape (n_layers, T)", diff.shape, (n_layers, T))
# verify manually
logits_all = resid @ W_U   # (n_layers, T, V)
expected = logits_all[..., correct] - logits_all[..., wrong]
check("matches manual", np.allclose(diff, expected), True)
# can be positive or negative
check("some positive some negative", diff.max() > 0 and diff.min() < 0, True)


### Head OV → Logit contribution
How much does head h's OV circuit move the logit for a specific output token, given input token?

In [ ]:
# 15d. OV-to-logit: for each (input_token, output_token) pair, score how much
# head h promotes output_token when attending to input_token
def ov_logit_scores(W_OV, W_U, W_E):
    """
    W_OV: (d, d)  — OV circuit for one head
    W_U:  (d, V)  — unembedding
    W_E:  (V, d)  — embedding
    Returns: (V, V) matrix where out[a, b] = how much attending to token a
             increases the logit for token b
             = W_E[a] @ W_OV @ W_U[:, b]
    """
    raise NotImplementedError()


In [ ]:
d, V = 8, 20
np.random.seed(33)
W_OV = np.random.randn(d, d)
W_U  = np.random.randn(d, V)
W_E  = np.random.randn(V, d)
scores = ov_logit_scores(W_OV, W_U, W_E)
check("shape (V, V)", scores.shape, (V, V))
# verify one entry manually
a, b = 2, 5
expected = W_E[a] @ W_OV @ W_U[:, b]
check("scores[a,b] = W_E[a] @ W_OV @ W_U[:,b]", np.isclose(scores[a,b], expected), True)


In [193]:
# 15e. Copy score: diagonal of ov_logit_scores — how much does each token promote itself?
def copy_scores(W_OV, W_U, W_E):
    """
    Returns: (V,) — copy score per token. Large positive = copy head for that token.
    """
    C_OV = np.einsum("am,mn,nb",W_E, W_OV, W_U)
    
    return np.diag(C_OV)


In [194]:
d, V = 8, 20
np.random.seed(34)
W_E  = np.random.randn(V, d); W_E /= np.linalg.norm(W_E, axis=1, keepdims=True)
W_U  = W_E.T                  # tied embeddings → copy circuit
W_OV_copy = np.eye(d) * 3.0   # identity OV = pure copy
W_OV_rand = np.random.randn(d, d) * 0.1

scores_copy = copy_scores(W_OV_copy, W_U, W_E)
scores_rand = copy_scores(W_OV_rand, W_U, W_E)
check("shape (V,)", scores_copy.shape, (V,))
check("copy head has positive mean copy score", scores_copy.mean() > 0, True)
check("copy head dominates random head", scores_copy.mean() > scores_rand.mean(), True)


  [PASS] shape (V,)
  [PASS] copy head has positive mean copy score
  [PASS] copy head dominates random head


### Attribution Patching
Linear approximation: effect of patching ≈ (clean − corrupt) · gradient

In [195]:
# 15f. Attribution patching — score each (layer, head, position) component
def attribution_patch_scores(head_outputs_clean, head_outputs_corrupt, gradients):
    """
    head_outputs_clean:   (n_layers, H, T, d) clean run head outputs
    head_outputs_corrupt: (n_layers, H, T, d) corrupt run head outputs
    gradients:            (n_layers, H, T, d) dMetric/d(head_output)
    Returns: (n_layers, H, T) attribution score per component
             score = sum_d (clean - corrupt) * grad
    """
    return ((head_outputs_clean - head_outputs_corrupt) * gradients).sum(axis=-1)


In [196]:
n_layers, H, T, d = 3, 4, 6, 8
np.random.seed(35)
clean   = np.random.randn(n_layers, H, T, d)
corrupt = np.random.randn(n_layers, H, T, d)
grads   = np.random.randn(n_layers, H, T, d)
scores  = attribution_patch_scores(clean, corrupt, grads)
check("shape (n_layers, H, T)", scores.shape, (n_layers, H, T))
# verify one entry manually
expected_00 = np.sum((clean[0,0] - corrupt[0,0]) * grads[0,0], axis=-1)  # (T,)
check("matches manual for layer 0 head 0", np.allclose(scores[0,0], expected_00), True)
# scores can be positive or negative
check("has positive and negative values", scores.max() > 0 and scores.min() < 0, True)


  [PASS] shape (n_layers, H, T)
  [PASS] matches manual for layer 0 head 0
  [PASS] has positive and negative values


### Solutions — Section 15

In [ ]:
def logit_lens(resid_stream, W_U):
    logits = resid_stream @ W_U   # (n_layers, T, V)
    probs  = softmax_axis(logits, axis=-1)
    return logits, probs

def top_token_per_layer(resid_stream, W_U):
    logits = resid_stream @ W_U   # (n_layers, T, V)
    return np.argmax(logits, axis=-1)   # (n_layers, T)

def logit_diff_lens(resid_stream, W_U, correct_idx, wrong_idx):
    logits = resid_stream @ W_U   # (n_layers, T, V)
    return logits[..., correct_idx] - logits[..., wrong_idx]

def ov_logit_scores(W_OV, W_U, W_E):
    return W_E @ W_OV @ W_U   # (V,d) @ (d,d) @ (d,V) → (V,V)

def copy_scores(W_OV, W_U, W_E):
    return np.diag(ov_logit_scores(W_OV, W_U, W_E))   # (V,)

def attribution_patch_scores(head_outputs_clean, head_outputs_corrupt, gradients):
    return np.sum((head_outputs_clean - head_outputs_corrupt) * gradients, axis=-1)


---
## 16. Activation Patching (conceptual + numpy)

**Setup**: you have a clean run and a corrupt run through a tiny 2-layer model.
Each run caches residual stream snapshots at every layer.
Patching = swap one layer's residual from corrupt → clean, re-run the rest, measure recovery.

The model here is stripped to pure numpy: no softmax, no nonlinearity — just linear layers
so we can test patching in isolation without a full transformer.

In [ ]:
# Tiny linear "model": output = sum of residual stream contributions per layer
# resid[L] = resid[L-1] + W[L] @ resid[L-1]  (simplified residual block)
# final logit difference = resid[-1] @ (W_U[:, correct] - W_U[:, wrong])

def run_model(x0, Ws, W_U, correct, wrong):
    """
    x0: (d,) initial embedding
    Ws: list of n_layers (d,d) weight matrices
    W_U: (d, V) unembedding
    correct, wrong: int token indices
    Returns: resid_cache (n_layers+1, d), logit_diff scalar
    """
    resid = x0.copy()
    cache = [resid.copy()]
    for W in Ws:
        resid = resid + W @ resid   # residual connection
        cache.append(resid.copy())
    logit_diff = resid @ (W_U[:, correct] - W_U[:, wrong])
    return np.array(cache), logit_diff


In [ ]:
# 16a. Patch residual stream at one layer: replace corrupt cache[layer] with clean,
#      re-run all subsequent layers, return new logit difference.
def patch_residual(layer, cache_clean, cache_corrupt, Ws, W_U, correct, wrong):
    """
    layer: int — which layer snapshot to patch (index into cache, 0 = embedding)
    cache_clean:   (n_layers+1, d)
    cache_corrupt: (n_layers+1, d)
    Ws: list of n_layers (d,d)
    Returns: patched logit_diff scalar
    """
    raise NotImplementedError()


In [ ]:
d, V, n_layers = 8, 10, 3
np.random.seed(40)
Ws    = [np.random.randn(d,d) * 0.1 for _ in range(n_layers)]
W_U   = np.random.randn(d, V)
correct, wrong = 2, 7

x_clean   = np.random.randn(d)
x_corrupt = np.random.randn(d)

cache_clean,   ld_clean   = run_model(x_clean,   Ws, W_U, correct, wrong)
cache_corrupt, ld_corrupt = run_model(x_corrupt, Ws, W_U, correct, wrong)

print(f"clean logit diff:   {ld_clean:.4f}")
print(f"corrupt logit diff: {ld_corrupt:.4f}")

# patch at each layer — patching layer 0 (embedding) should fully recover clean
ld_patch_0 = patch_residual(0, cache_clean, cache_corrupt, Ws, W_U, correct, wrong)
ld_patch_last = patch_residual(n_layers, cache_clean, cache_corrupt, Ws, W_U, correct, wrong)

check("patch at layer 0 = full clean run", np.isclose(ld_patch_0, ld_clean, atol=1e-8), True)
check("patch at last layer = clean logit diff", np.isclose(ld_patch_last, ld_clean, atol=1e-8), True)
check("patch at layer 0 != corrupt", not np.isclose(ld_patch_0, ld_corrupt, atol=1e-4), True)


In [ ]:
# 16b. Scan all layers — compute patched logit diff at every layer,
#      return (n_layers+1,) array of patched logit diffs.
#      This is the "patching curve" — shows where information lives.
def patching_curve(cache_clean, cache_corrupt, Ws, W_U, correct, wrong):
    """Returns (n_layers+1,) patched logit diff for each patch site"""
    raise NotImplementedError()


In [ ]:
curve = patching_curve(cache_clean, cache_corrupt, Ws, W_U, correct, wrong)
check("shape (n_layers+1,)", curve.shape, (n_layers+1,))
check("layer 0 patch = clean",      np.isclose(curve[0], ld_clean,   atol=1e-8), True)
check("last layer patch = clean",   np.isclose(curve[-1], ld_clean,  atol=1e-8), True)
# no patch = corrupt
_, ld_no_patch = run_model(x_corrupt, Ws, W_U, correct, wrong)
check("no patch = corrupt baseline", np.isclose(ld_no_patch, ld_corrupt, atol=1e-8), True)
print("\nPatching curve (logit diff at each patch site):")
for i, v in enumerate(curve):
    bar = "█" * int(abs(v) * 5)
    print(f"  layer {i}: {v:+.4f}  {bar}")


### Solutions — Section 16

In [ ]:
def patch_residual(layer, cache_clean, cache_corrupt, Ws, W_U, correct, wrong):
    # start from corrupt cache up to patch point, then swap in clean
    resid = cache_corrupt[layer].copy()
    resid = cache_clean[layer].copy()   # the patch: overwrite with clean
    # re-run all layers after the patch point
    for W in Ws[layer:]:
        resid = resid + W @ resid
    return resid @ (W_U[:, correct] - W_U[:, wrong])

def patching_curve(cache_clean, cache_corrupt, Ws, W_U, correct, wrong):
    n = len(Ws)
    return np.array([
        patch_residual(layer, cache_clean, cache_corrupt, Ws, W_U, correct, wrong)
        for layer in range(n + 1)
    ])


---
## 🔥 17. Indexing & Slicing — Predict the Output

Each cell: **predict before running**. The patterns that bit you on the test are here.

### 17a. Step and reverse slicing

In [187]:
# a[1:triples = [(l,h,scores[l,h]) for l in range(L) for h in range(H)]
#     return sorted(triples, key=lambda x: -x[2])6]

array([20, 30, 40, 50, 60])

In [172]:
a = np.array([10, 20, 30, 40, 50, 60])

print(a[::-1])        # reverse entire array
print(a[::2])         # every other element (0,2,4)
print(a[1::2])        # every other starting at index 1
print(a[-3:])         # last 3
print(a[:-2])         # all but last 2
print(a[-4:-1])       # index -4 up to (not including) -1


[60 50 40 30 20 10]
[10 30 50]
[20 40 60]
[40 50 60]
[10 20 30 40]
[30 40 50]


### 17b. 2D slicing — rows and columns

In [ ]:
A = np.arange(12).reshape(3, 4)
print("A =\n", A)
print()
print(A[:, ::-1])      # reverse columns — RIGHT to LEFT
print(A[::-1, :])      # reverse rows — BOTTOM to TOP
print(A[::-1, ::-1])   # flip both
print(A[:, -2:])       # last 2 columns
print(A[-2:, :])       # last 2 rows
print(A[1:, 1:])       # submatrix from (1,1)


### 17c. argsort + slice — the pattern that bit you

In [ ]:
a = np.array([3., 1., 4., 1., 5., 9., 2., 6.])

# argsort gives ASCENDING order
idx_asc = np.argsort(a)
print("ascending idx:", idx_asc)
print("ascending vals:", a[idx_asc])

# top-3 indices (ascending order of value — WRONG for most purposes)
print("top-3 wrong order:", a[idx_asc[-3:]])

# top-3 indices DESCENDING — add [::-1]
idx_top3 = idx_asc[-3:][::-1]
print("top-3 correct:    ", a[idx_top3])

# 2D version — per row
A = np.random.randn(4, 6)
idx_2d = np.argsort(A, axis=1)[:, -3:][:, ::-1]   # top-3 per row, descending
vals_2d = np.take_along_axis(A, idx_2d, axis=1)
print("2D top-3 sorted descending per row:", np.all(vals_2d[:,:-1] >= vals_2d[:,1:]))


### 17d. Fancy indexing — selecting specific rows/columns

In [ ]:
A = np.arange(20).reshape(4, 5)
print("A =\n", A)

# select specific rows
print(A[[0, 2]])          # rows 0 and 2 → shape (2,5)

# select specific columns
print(A[:, [1, 3]])       # columns 1 and 3 → shape (4,2)

# select specific (row, col) PAIRS — NOT a submatrix
rows = [0, 1, 2]
cols = [4, 3, 2]
print(A[rows, cols])      # [A[0,4], A[1,3], A[2,2]] → shape (3,)

# for a submatrix from index lists, use np.ix_
print(A[np.ix_([0,2], [1,3])])   # 2x2 submatrix → shape (2,2)


### 17e. Boolean indexing

In [ ]:
a = np.array([1., -2., 3., -4., 5.])

# select positive values
print(a[a > 0])            # [1. 3. 5.] — 1D result regardless of input shape

# zero out negatives
b = a.copy()
b[b < 0] = 0.
print(b)

# 2D — boolean mask selects elements, flattens
A = np.array([[1., -2.], [3., -4.]])
print(A[A > 0])            # [1. 3.] — flattened!

# use mask to set values
A[A < 0] = 0.
print(A)


### 17f. Adding dimensions: None / np.newaxis

In [ ]:
a = np.array([1., 2., 3.])
print("a.shape:", a.shape)          # (3,)

print(a[:, None].shape)             # (3,1) — column vector
print(a[None, :].shape)             # (1,3) — row vector
print(a[:, np.newaxis].shape)       # same as [:, None]

# useful for broadcasting outer product without einsum
b = np.array([4., 5.])
print((a[:, None] * b[None, :]).shape)   # (3,2) outer product
print(np.allclose(a[:, None] * b, np.outer(a, b)))   # True


### 17g. Ellipsis `...` — index last/first dim of any shape

In [ ]:
A = np.random.randn(3, 4, 5, 6)

print(A[0].shape)          # (4,5,6) — same as A[0,:,:,:]
print(A[..., 0].shape)     # (3,4,5) — last dim, same as A[:,:,:,0]
print(A[0, ..., 0].shape)  # (4,5)   — first and last dim fixed

# useful for batched operations
# e.g. residual stream (n_layers, T, d) — get all layer 0 vectors:
resid = np.random.randn(4, 6, 8)
print(resid[0].shape)        # (6,8) — layer 0, all positions, all features
print(resid[:, -1].shape)    # (4,8) — all layers, last position
print(resid[:, -1, :].shape) # (4,8) — same, explicit


### 17h. Diagonal indexing

In [ ]:
A = np.arange(9.).reshape(3,3)
print("A =\n", A)

print(np.diag(A))              # [0. 4. 8.] — extract main diagonal
print(np.diag(np.diag(A)))     # diagonal matrix from those values
print(A[range(3), range(3)])   # same as np.diag — fancy index pairs

# off-diagonal
print(np.diag(A, k=1))         # [1. 5.] — one above main diagonal
print(np.diag(A, k=-1))        # [3. 7.] — one below main diagonal

# for induction: extract stripe A[n+i, i+1]
n = 3; T = 2*n
B = np.random.randn(T, T)
stripe = B[np.arange(n,T-1), np.arange(1,n)]   # A[n+i, i+1] for i in 0..n-2
print("stripe shape:", stripe.shape)            # (n-1,)


### 17i. put_along_axis — set values at indices

In [ ]:
# put_along_axis: inverse of take_along_axis
A = np.ones((3, 6))
idx = np.array([[0, 2, 4],   # row 0: zero out cols 0,2,4
                [1, 3, 5],   # row 1: zero out cols 1,3,5
                [0, 1, 2]])  # row 2: zero out cols 0,1,2

np.put_along_axis(A, idx, 0., axis=1)
print(A)

# common pattern: zero out non-top-k
B = np.abs(np.random.randn(3, 8))
B_sparse = B.copy()
k = 3
idx_zero = np.argsort(B_sparse, axis=1)[:, :-k]   # indices of BOTTOM (n-k)
np.put_along_axis(B_sparse, idx_zero, 0., axis=1)
print("non-zero per row:", np.sum(B_sparse > 0, axis=1))  # should be [k,k,k]


### 17j. Combined — implement from scratch using slicing

In [ ]:
# Given A (T, d), implement each in ONE line using slicing only (no loops):

def flip_columns(A):
    # reverse the column order
    raise NotImplementedError()

def last_k_rows(A, k):
    # return last k rows
    raise NotImplementedError()

def every_other_col(A):
    # return columns 0, 2, 4, ... (even indices)
    raise NotImplementedError()

def upper_left(A, k):
    # return top-left k×k submatrix
    raise NotImplementedError()

def anti_diagonal(A):
    # return the anti-diagonal (top-right to bottom-left) of a square matrix
    # hint: flip columns first, then take diagonal
    raise NotImplementedError()


In [ ]:
A = np.arange(20.).reshape(4, 5)
check("flip_columns",   np.allclose(flip_columns(A), A[:, ::-1]), True)
check("last_k_rows",    np.allclose(last_k_rows(A, 2), A[-2:, :]), True)
check("every_other_col",np.allclose(every_other_col(A), A[:, ::2]), True)

B = np.arange(16.).reshape(4,4)
check("upper_left",     np.allclose(upper_left(B, 2), B[:2, :2]), True)
check("anti_diagonal",  np.allclose(anti_diagonal(B), np.diag(B[:, ::-1])), True)


### Solutions — Section 17

In [ ]:
def flip_columns(A):        return A[:, ::-1]
def last_k_rows(A, k):      return A[-k:, :]
def every_other_col(A):     return A[:, ::2]
def upper_left(A, k):       return A[:k, :k]
def anti_diagonal(A):       return np.diag(A[:, ::-1])


---
## Solutions
Only look after attempting. Expand cells below.

In [ ]:
# ── SOLUTIONS ──────────────────────────────────────────────────────────────────

def softmax_1d(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

def softmax_2d(Z):
    Z = Z - Z.max(axis=-1, keepdims=True); e = np.exp(Z); return e / e.sum(axis=-1, keepdims=True)

def softmax_axis(Z, axis=-1):
    Z = Z - Z.max(axis=axis, keepdims=True); e = np.exp(Z); return e / e.sum(axis=axis, keepdims=True)

def log_softmax(z):
    z = z - z.max(); return z - np.log(np.sum(np.exp(z)))

def cross_entropy(logits, target_idx):
    return -log_softmax(logits)[target_idx]

def cross_entropy_batch(logits, targets):
    ls = logits - logits.max(axis=-1, keepdims=True)
    ls = ls - np.log(np.sum(np.exp(ls), axis=-1, keepdims=True))
    return -np.mean(ls[np.arange(len(targets)), targets])

def softmax_jacobian(z):
    p = softmax_1d(z); return np.diag(p) - np.outer(p, p)

def softmax_grad(z, dL_dp):
    p = softmax_1d(z); return p * (dL_dp - (dL_dp * p).sum())

def svd_reconstruct(A, rank):
    U, S, Vh = np.linalg.svd(A, full_matrices=False)
    return (U[:, :rank] * S[:rank]) @ Vh[:rank, :]

def effective_rank(S, threshold=0.9):
    s2 = S**2; return int(np.searchsorted(np.cumsum(s2)/s2.sum(), threshold) + 1)

def batch_effective_rank(W_OVs, threshold=0.9):
    _, S, _ = np.linalg.svd(W_OVs, full_matrices=False)
    return np.array([effective_rank(S[h], threshold) for h in range(len(W_OVs))])

def top_eigenvector(M):
    vals, vecs = np.linalg.eigh(M); return vecs[:, -1]

def copy_head_score(W_OV, W_U, W_E):
    M = W_E @ W_OV @ W_U @ W_E.T  # wait: actually W_OV@W_U gives (d,V), W_E@that gives (V,V)
    # score = max real eigenvalue of W_OV @ W_U (d,V) — but that's not square
    # correct: eigenvalues of W_E @ W_OV @ W_U (V,d)@(d,d)@(d,V) — use (d,d) circuit
    circuit = W_OV @ W_U @ W_E  # (d,d)
    return np.real(np.linalg.eigvals(circuit)).max()

def sym_anti(W_QK):
    return (W_QK + W_QK.T) / 2, (W_QK - W_QK.T) / 2

def outer(a, b):       return np.einsum("i,j->ij", a, b)
def inner(a, b):       return np.einsum("i,i->", a, b)
def matrix_trace(A):   return np.einsum("ii->", A)

def batched_matmul(A, B): return np.einsum("hmk,hkn->hmn", A, B)
def bilinear(x, M, y):   return np.einsum("i,ij,j->", x, M, y)

def circuit_matrices(W_Qs, W_Ks, W_Vs, W_Os):
    W_QKs = np.einsum("hia,hja->hij", W_Qs, W_Ks)
    W_OVs = np.einsum("hia,haj->hij", W_Vs, W_Os)
    return W_QKs, W_OVs

def topk_1d(a, k):
    idx = np.argsort(a)[-k:][::-1]; return idx, a[idx]

def topk_2d(A, k):
    idx = np.argsort(A, axis=1)[:, -k:][:, ::-1]
    return idx, np.take_along_axis(A, idx, axis=1)

def zero_non_topk(A, k):
    out = A.copy()
    idx_zero = np.argsort(A, axis=1)[:, :-k]
    np.put_along_axis(out, idx_zero, 0.0, axis=1)
    return out

def attention(Q, K, V):
    dk = Q.shape[-1]; return softmax_2d(Q @ K.T / np.sqrt(dk)) @ V

def causal_attention(Q, K, V):
    T, dk = Q.shape
    scores = Q @ K.T / np.sqrt(dk)
    mask = np.triu(np.ones((T,T)), k=1).astype(bool)
    scores[mask] = -np.inf
    return softmax_2d(scores) @ V

def multihead_attention(X, W_Qs, W_Ks, W_Vs, W_Os):
    out = np.zeros((X.shape[0], X.shape[1]))
    for h in range(W_Qs.shape[0]):
        Q=X@W_Qs[h]; K=X@W_Ks[h]; V=X@W_Vs[h]
        out += attention(Q,K,V) @ W_Os[h]
    return out

def entropy(p):
    p = p[p > 0]; return -np.sum(p * np.log(p))

def mean_head_entropy(logits_per_layer):
    arr = np.array(logits_per_layer)
    A = softmax_axis(arr, axis=-1)
    ent = -np.sum(A * np.log(np.clip(A, 1e-10, None)), axis=-1)
    return ent.mean(axis=-1)

def most_uniform_head(logits_per_layer):
    H = mean_head_entropy(logits_per_layer)
    idx = np.unravel_index(np.argmax(H), H.shape)
    return idx

def induction_score_single(A, n):
    return np.mean([A[n+i, i+1] for i in range(n-1)])

def score_all_induction(weights_per_layer, n):
    arr = np.array(weights_per_layer)
    L, H = arr.shape[:2]
    return np.array([[induction_score_single(arr[l,h], n) for h in range(H)] for l in range(L)])

def rank_induction_heads(weights_per_layer, n):
    scores = score_all_induction(weights_per_layer, n)
    L, H = scores.shape
    triples = [(l,h,scores[l,h]) for l in range(L) for h in range(H)]
    return sorted(triples, key=lambda x: -x[2])

def sae_encode(x, W_enc, b_enc):
    return np.maximum(0, x @ W_enc + b_enc)

def sae_topk_decode(acts, W_dec, b_dec, k):
    idx = np.argsort(acts, axis=1)[:, -k:][:, ::-1]
    vals = np.take_along_axis(acts, idx, axis=1)
    sparse = np.zeros_like(acts)
    np.put_along_axis(sparse, idx, vals, axis=1)
    recon = sparse @ W_dec + b_dec
    return recon, idx, vals

def sae_full(x, W_enc, b_enc, W_dec, b_dec, k):
    acts = sae_encode(x, W_enc, b_enc)
    recon, idx, vals = sae_topk_decode(acts, W_dec, b_dec, k)
    mse = float(np.mean((x - recon)**2))
    return recon, idx, vals, mse

def composition_strength(W_OV, W_proj):
    num = np.linalg.norm(W_OV @ W_proj, 'fro')
    denom = np.linalg.norm(W_OV, 'fro') * np.linalg.norm(W_proj, 'fro')
    return num / denom

def k_composed_qk(W_OV_L1, W_Q_L2, W_K_L2):
    return W_Q_L2 @ (W_OV_L1 @ W_K_L2).T

def self_reinforce_score(W_OV_L1, W_OV_L2, W_enc, W_dec):
    W_virt = W_OV_L1 @ W_OV_L2
    return np.einsum("fd,de,ef->f", W_dec, W_virt, W_enc)

def linear_attention(Q, K, V):
    return Q @ (K.T @ V)

def causal_linear_attention(Q, K, V):
    T, dk = Q.shape; dv = V.shape[1]
    out = np.zeros((T, dv))
    KV = np.zeros((dk, dv))
    for t in range(T):
        KV += np.outer(K[t], V[t])
        out[t] = Q[t] @ KV
    return out


---
## 🔥 14. NumPy / Python Gotchas
Each cell has a broken or ambiguous snippet. Fix it or predict the output before running.

### G1. np.ndarray vs np.array

In [ ]:
# What is wrong here? Fix it.
data = [1, 2, 3, 4]
# x = np.ndarray(data)   # BUG
x = None  # fix this line
print(x, x.shape if x is not None else "")


In [ ]:
# Answer:
x = np.array(data)          # np.ndarray(data) interprets data as SHAPE, not content
print(x, x.shape)           # np.ndarray((4,)) would give uninitialized (4,) array


### G2. View vs Copy — silent mutation bug

In [ ]:
# Predict: what does a look like after this runs?
a = np.array([1., 2., 3., 4., 5.])
b = a[1:4]
b[0] = 99.
print("a =", a)   # predict before running


In [ ]:
# Fix: make b a copy so a is not modified
a = np.array([1., 2., 3., 4., 5.])
b = None  # fix this line
b[0] = 99.
print("a =", a)   # should still be [1,2,3,4,5]


In [ ]:
# Answer:
a = np.array([1., 2., 3., 4., 5.])
b = a[1:4].copy()
b[0] = 99.
print("a =", a)   # [1. 2. 3. 4. 5.] — unchanged


### G3. Shape (3,) vs (3,1) — matmul failure

In [ ]:
# This will raise an error. Why? Fix it two ways.
A = np.random.randn(4, 3)
v = np.array([1., 2., 3.])

# way 1: make v a column vector
# way 2: use @

try:
    out = A @ v        # does this work?
    print("shape:", out.shape)
except Exception as e:
    print("Error:", e)


In [ ]:
# Answer: A @ v actually WORKS for (4,3) @ (3,) → (4,) — numpy handles this case
# The dangerous case is when you need (4,3) @ (3,1) → (4,1) for batch consistency

A = np.random.randn(4, 3)
v = np.array([1., 2., 3.])
print((A @ v).shape)          # (4,)  — fine for most uses
print((A @ v[:,None]).shape)  # (4,1) — column vector output
print((A @ v.reshape(-1,1)).shape)  # same


### G4. argsort direction

In [170]:
# What order does this return? Is it ascending or descending?
a = np.array([3., 1., 4., 1., 5., 9., 2., 6.])
top3_idx = np.argsort(a)[-3:]
print("indices:", top3_idx)
print("values:", a[top3_idx])   # are these sorted?


indices: [4 7 5]
values: [5. 6. 9.]


In [171]:
# Fix: get top-3 indices sorted descending
a = np.array([3., 1., 4., 1., 5., 9., 2., 6.])
top3_idx = np.argsort(a)[-3:][::-1]   # reverse to get descending
print("indices:", top3_idx)
print("values:", a[top3_idx])          # [9, 6, 5]


indices: [5 7 4]
values: [9. 6. 5.]


### G5. Effective rank: cumsum(S) vs cumsum(S**2)

In [ ]:
# Both of these compile and run. Which is correct for effective rank?
S = np.array([5., 3., 1., 0.1])

# version A
cumA = np.cumsum(S) / np.sum(S)
rankA = np.searchsorted(cumA, 0.9) + 1

# version B
cumB = np.cumsum(S**2) / np.sum(S**2)
rankB = np.searchsorted(cumB, 0.9) + 1

print(f"rank A (wrong): {rankA}")
print(f"rank B (correct): {rankB}")
# Frobenius norm = sqrt(sum of squared singular values) → use S**2


### G6. np.linalg.svd returns Vh (already transposed)

In [ ]:
# Spot the bug: reconstruct A from SVD
A = np.random.randn(4, 5)
U, S, Vh = np.linalg.svd(A, full_matrices=False)

# BUG version:
A_bug = U * S @ Vh.T   # wrong — Vh is already transposed

# Correct version:
A_ok  = U * S @ Vh     # Vh is V^T, so this is U S V^T = A

print("bug error:  ", np.linalg.norm(A - A_bug))
print("correct err:", np.linalg.norm(A - A_ok))


### G7. axis=0 collapses ROWS (operates down columns)

In [ ]:
A = np.array([[1., 2., 3.],
              [4., 5., 6.]])   # shape (2, 3)

print("sum axis=0:", np.sum(A, axis=0).shape, np.sum(A, axis=0))  # (3,) — column sums
print("sum axis=1:", np.sum(A, axis=1).shape, np.sum(A, axis=1))  # (2,) — row sums

# Which do you want for "sum over sequence positions to get one value per feature"?
# A is (T, d_model) → sum over T → axis=0 → shape (d_model,)
T, d = 5, 4
X = np.random.randn(T, d)
mean_over_tokens = np.mean(X, axis=0)   # shape (d,) ← this one
print("mean over tokens shape:", mean_over_tokens.shape)


### G8. * is element-wise, @ is matmul — Julia trap

In [ ]:
A = np.array([[1.,2.],[3.,4.]])
B = np.array([[5.,6.],[7.,8.]])

print("A * B (element-wise):\n", A * B)   # Hadamard
print("A @ B (matmul):\n",       A @ B)   # matrix multiply

# In Julia: A * B = matmul, A .* B = element-wise — OPPOSITE of NumPy
# Common bug: writing A * B when you mean A @ B


### G9. Integer dtype overflow (silent!)

In [ ]:
# Predict the output:
a = np.array([200], dtype=np.int8)
print(a * 2)    # what happens?
print()

# Safe version:
b = np.array([200], dtype=np.int32)
print(b * 2)

# Lesson: default np.array([...]) is int64 for integers, float64 for floats
# int8 max is 127 — overflow wraps silently


### G10. Nested list trap (Python)

In [197]:
# Predict: what does grid look like after this?
grid = [[0] * 3] * 3
grid[0][0] = 99
print(grid)   # ALL rows changed — they're the same object

# Fix:
grid_fixed = [[0] * 3 for _ in range(3)]
grid_fixed[0][0] = 99
print(grid_fixed)   # only row 0 changed


[[99, 0, 0], [99, 0, 0], [99, 0, 0]]
[[99, 0, 0], [0, 0, 0], [0, 0, 0]]


### G11. take_along_axis — the right way to gather by index

In [ ]:
# Given a (T, d) array and a (T, k) index array, extract the indexed values.
# Common wrong approach:
T, d, k = 4, 8, 3
np.random.seed(42)
A = np.random.randn(T, d)
idx = np.argsort(A, axis=1)[:, -k:]   # top-k indices per row

# Wrong: fancy indexing with 2D idx doesn't do what you want
# A[idx]  would select ROWS not elements

# Correct:
vals = np.take_along_axis(A, idx, axis=1)   # shape (T, k)
print("vals shape:", vals.shape)
print("manually verify row 0:", np.allclose(vals[0], A[0, idx[0]]))


### G12. log(0) = -inf in entropy — use np.clip

In [ ]:
# Entropy of a distribution with zeros — naive version crashes
p = np.array([0.5, 0.5, 0.0, 0.0])

# Bug: 0 * log(0) = nan in numpy (should be 0 by convention)
naive = -np.sum(p * np.log(p))
print("naive:", naive)   # nan — because log(0) = -inf, 0 * -inf = nan

# Fix 1: clip
safe1 = -np.sum(p * np.log(np.clip(p, 1e-10, None)))
print("clipped:", safe1)

# Fix 2: only sum nonzero
safe2 = -np.sum(p[p>0] * np.log(p[p>0]))
print("nonzero only:", safe2)
